# Proteomics MAS Orchestrator (improved)

This notebook extends the original MAS framework so it can:

- parse per-drug markdown cards (`.md`) into a standardized JSON package
- tolerate empty Hallmark / GO sections when no significant pathways are present
- validate and normalize input more strictly
- run a 3-agent pipeline (`Analyzer -> Reviewer -> Supervisor`)
- batch process many drug cards
- save one JSON result and one markdown summary per drug


## 1. Imports

In [1]:

from __future__ import annotations

import json
import math
import os
import re
from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Iterable

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None  # type: ignore


## 2. Input schema, normalization, and validation

In [2]:

GENE_ROW_REQUIRED_FIELDS = ["gene_symbol", "log2FC", "p_value_adj"]
PATHWAY_ROW_REQUIRED_FIELDS = ["Term", "NES", "FDR"]

DRUG_PACKAGE_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "required": ["basic_info", "protein_summary", "pathway_summary"],
    "properties": {
        "basic_info": {
            "type": "object",
            "required": ["drug_name"],
            "properties": {
                "drug_name": {"type": "string"},
                "target": {"type": "array", "items": {"type": "string"}},
                "pathway": {"type": "array", "items": {"type": "string"}},
                "research_area": {"type": "array", "items": {"type": "string"}},
                "known_moa": {"type": "string"},
                "notes": {"type": "string"},
            },
        },
        "protein_summary": {
            "type": "object",
            "properties": {
                "top_up": {"type": "array", "items": {"type": "object"}},
                "top_down": {"type": "array", "items": {"type": "object"}},
                "significant_counts": {
                    "type": "object",
                    "properties": {
                        "up": {"type": "number"},
                        "down": {"type": "number"},
                        "total": {"type": "number"},
                    },
                },
            },
        },
        "pathway_summary": {
            "type": "object",
            "properties": {
                "hallmark_up": {"type": "array", "items": {"type": "object"}},
                "hallmark_down": {"type": "array", "items": {"type": "object"}},
                "go_bp_up": {"type": "array", "items": {"type": "object"}},
                "go_bp_down": {"type": "array", "items": {"type": "object"}},
                "reactome": {"type": "array", "items": {"type": "object"}},
                "kegg": {"type": "array", "items": {"type": "object"}},
                "empty_sections": {"type": "array", "items": {"type": "string"}},
            },
        },
        "supporting_context": {
            "type": "object",
            "properties": {
                "similar_drugs": {"type": "array", "items": {"type": "string"}},
                "cluster_membership": {"type": "string"},
                "qc_notes": {"type": "string"},
                "external_validation_clues": {"type": "array", "items": {"type": "string"}},
            },
        },
        "metadata": {
            "type": "object",
            "properties": {
                "source_file": {"type": "string"},
                "input_format": {"type": "string"},
                "created_by": {"type": "string"},
                "version": {"type": "string"},
            },
        },
    },
}


def make_empty_drug_package() -> Dict[str, Any]:
    return {
        "basic_info": {
            "drug_name": "",
            "target": [],
            "pathway": [],
            "research_area": [],
            "known_moa": "",
            "notes": "",
        },
        "protein_summary": {
            "top_up": [],
            "top_down": [],
            "significant_counts": {"up": 0, "down": 0, "total": 0},
        },
        "pathway_summary": {
            "hallmark_up": [],
            "hallmark_down": [],
            "go_bp_up": [],
            "go_bp_down": [],
            "reactome": [],
            "kegg": [],
            "empty_sections": [],
        },
        "supporting_context": {
            "similar_drugs": [],
            "cluster_membership": "",
            "qc_notes": "",
            "external_validation_clues": [],
        },
        "metadata": {
            "source_file": "",
            "input_format": "",
            "created_by": "proteomics_mas_orchestrator_improved",
            "version": "2.0",

            
        },
    }

def _is_missing_scalar(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and value.strip().lower() in {"", "na", "nan", "none", "null"}:
        return True
    return False

def _to_float_if_possible(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return value
    if not isinstance(value, str):
        return value
    s = value.strip()
    if s.lower() in {"", "na", "nan", "none", "null"}:
        return None
    try:
        if re.fullmatch(r"[-+]?\d+", s):
            return int(s)
        return float(s)
    except ValueError:
        return s

def _split_semicolon_field(value: Any) -> List[str]:
    if _is_missing_scalar(value):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if not _is_missing_scalar(x)]
    parts = re.split(r"[;|]", str(value))
    return [p.strip() for p in parts if p.strip()]

def _clean_gene_rows(rows: List[Dict[str, Any]], *, drop_crap: bool = True, drop_missing_gene: bool = True) -> List[Dict[str, Any]]:
    cleaned: List[Dict[str, Any]] = []
    for row in rows:
        row2 = {k: _to_float_if_possible(v) for k, v in row.items()}
        protein = str(row2.get("protein", "") or "").strip()
        gene = row2.get("gene_symbol")

        if drop_crap and protein.lower().startswith("crap-"):
            continue
        if drop_missing_gene and _is_missing_scalar(gene):
            continue

        if gene is not None:
            row2["gene_symbol"] = str(gene).strip()

        cleaned.append(row2)
    return cleaned

def _clean_pathway_rows(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [{k: _to_float_if_possible(v) for k, v in row.items()} for row in rows]

def normalize_drug_package(drug_package: Dict[str, Any]) -> Dict[str, Any]:
    pkg = make_empty_drug_package()

    basic_in = drug_package.get("basic_info", {}) or {}
    pkg["basic_info"]["drug_name"] = str(basic_in.get("drug_name", "") or "").strip()
    pkg["basic_info"]["target"] = _split_semicolon_field(basic_in.get("target", []))
    pkg["basic_info"]["pathway"] = _split_semicolon_field(basic_in.get("pathway", []))
    pkg["basic_info"]["research_area"] = _split_semicolon_field(basic_in.get("research_area", []))
    pkg["basic_info"]["known_moa"] = str(basic_in.get("known_moa", "") or "").strip()
    pkg["basic_info"]["notes"] = str(basic_in.get("notes", "") or "").strip()

    prot_in = drug_package.get("protein_summary", {}) or {}
    pkg["protein_summary"]["top_up"] = _clean_gene_rows(prot_in.get("top_up", []) or [])
    pkg["protein_summary"]["top_down"] = _clean_gene_rows(prot_in.get("top_down", []) or [])
    sig = prot_in.get("significant_counts", {}) or {}
    up_count = int(sig.get("up", len(pkg["protein_summary"]["top_up"])) or 0)
    down_count = int(sig.get("down", len(pkg["protein_summary"]["top_down"])) or 0)
    total_count = int(sig.get("total", up_count + down_count) or 0)
    pkg["protein_summary"]["significant_counts"] = {
        "up": up_count,
        "down": down_count,
        "total": total_count,
    }

    pw_in = drug_package.get("pathway_summary", {}) or {}
    pkg["pathway_summary"]["hallmark_up"] = _clean_pathway_rows(pw_in.get("hallmark_up", []) or [])
    pkg["pathway_summary"]["hallmark_down"] = _clean_pathway_rows(pw_in.get("hallmark_down", []) or [])
    pkg["pathway_summary"]["go_bp_up"] = _clean_pathway_rows(pw_in.get("go_bp_up", []) or [])
    pkg["pathway_summary"]["go_bp_down"] = _clean_pathway_rows(pw_in.get("go_bp_down", []) or [])
    pkg["pathway_summary"]["reactome"] = _clean_pathway_rows(pw_in.get("reactome", []) or [])
    pkg["pathway_summary"]["kegg"] = _clean_pathway_rows(pw_in.get("kegg", []) or [])

    empty_sections = set(pw_in.get("empty_sections", []) or [])
    for section_name in ["hallmark_up", "hallmark_down", "go_bp_up", "go_bp_down"]:
        if not pkg["pathway_summary"][section_name]:
            empty_sections.add(section_name)
    pkg["pathway_summary"]["empty_sections"] = sorted(empty_sections)

    ctx_in = drug_package.get("supporting_context", {}) or {}
    pkg["supporting_context"]["similar_drugs"] = _split_semicolon_field(ctx_in.get("similar_drugs", []))
    pkg["supporting_context"]["cluster_membership"] = str(ctx_in.get("cluster_membership", "") or "").strip()
    pkg["supporting_context"]["qc_notes"] = str(ctx_in.get("qc_notes", "") or "").strip()
    pkg["supporting_context"]["external_validation_clues"] = _split_semicolon_field(
        ctx_in.get("external_validation_clues", [])
    )

    meta_in = drug_package.get("metadata", {}) or {}
    pkg["metadata"]["source_file"] = str(meta_in.get("source_file", "") or "").strip()
    pkg["metadata"]["input_format"] = str(meta_in.get("input_format", "") or "").strip()
    pkg["metadata"]["created_by"] = str(meta_in.get("created_by", pkg["metadata"]["created_by"]) or "").strip()
    pkg["metadata"]["version"] = str(meta_in.get("version", pkg["metadata"]["version"]) or "").strip()

    return pkg

def validate_gene_rows(rows: List[Dict[str, Any]], row_name: str) -> None:
    if not isinstance(rows, list):
        raise ValueError(f"{row_name} must be a list.")
    for idx, row in enumerate(rows):
        if not isinstance(row, dict):
            raise ValueError(f"{row_name}[{idx}] must be a dictionary.")
        for field in GENE_ROW_REQUIRED_FIELDS:
            if field not in row:
                raise ValueError(f"Missing field {row_name}[{idx}].{field}")

def validate_pathway_rows(rows: List[Dict[str, Any]], row_name: str) -> None:
    if not isinstance(rows, list):
        raise ValueError(f"{row_name} must be a list.")
    for idx, row in enumerate(rows):
        if not isinstance(row, dict):
            raise ValueError(f"{row_name}[{idx}] must be a dictionary.")
        for field in PATHWAY_ROW_REQUIRED_FIELDS:
            if field not in row:
                raise ValueError(f"Missing field {row_name}[{idx}].{field}")


def validate_drug_package(drug_package: Dict[str, Any]) -> None:
    if not isinstance(drug_package, dict):
        raise ValueError("Drug package must be a dictionary.")

    for field in ["basic_info", "protein_summary", "pathway_summary"]:
        if field not in drug_package:
            raise ValueError(f"Missing required top-level field: {field}")

    basic_info = drug_package["basic_info"]
    if not isinstance(basic_info, dict):
        raise ValueError("basic_info must be a dictionary.")
    drug_name = str(basic_info.get("drug_name", "") or "").strip()
    if not drug_name:
        raise ValueError("Missing required field: basic_info.drug_name")

    protein_summary = drug_package["protein_summary"]
    if not isinstance(protein_summary, dict):
        raise ValueError("protein_summary must be a dictionary.")
    validate_gene_rows(protein_summary.get("top_up", []), "protein_summary.top_up")
    validate_gene_rows(protein_summary.get("top_down", []), "protein_summary.top_down")

    pathway_summary = drug_package["pathway_summary"]
    if not isinstance(pathway_summary, dict):
        raise ValueError("pathway_summary must be a dictionary.")
    for name in ["hallmark_up", "hallmark_down", "go_bp_up", "go_bp_down", "reactome", "kegg"]:
        validate_pathway_rows(pathway_summary.get(name, []), f"pathway_summary.{name}")


def prepare_drug_package(drug_package: Dict[str, Any]) -> Dict[str, Any]:
    pkg = normalize_drug_package(drug_package)
    validate_drug_package(pkg)
    return pkg


## 3. Agent prompts

In [3]:

ANALYZER_PROMPT = """
You are the Analyzer in a role-based multi-agent system for proteomics result interpretation.

Your role:
- identify the strongest data-driven biological patterns from a single-drug proteomics result package
- summarize dominant proteomic signatures
- identify major up-regulated and down-regulated programs
- propose preliminary mechanistic interpretations
- highlight unexpected but well-supported findings

Input may include:
- basic drug information
- top up-regulated proteins
- top down-regulated proteins
- Hallmark pathway enrichment (up and/or down; sometimes empty)
- GO BP up-regulated pathways (sometimes empty)
- GO BP down-regulated pathways (sometimes empty)
- optional context such as similar drugs or cluster membership

Important interpretation rule:
If a pathway section is empty, treat it as "no significant enriched pathways were supplied for that section".
Do NOT treat an empty section as a contradiction or data-quality failure by itself.

Strict rules:
1. Base reasoning primarily on the provided data.
2. Do not rely heavily on prior knowledge unless explicitly present in the input.
3. Any strong mechanistic statement should ideally be supported by BOTH protein-level and pathway-level evidence.
4. If only one evidence layer is present, phrase conclusions more cautiously.
5. Do not treat a single protein change as sufficient proof of a mechanism.
6. Distinguish dominant primary programs from broader secondary stress responses.
7. Be conservative and avoid overinterpretation.

Return valid JSON only with this structure:
{
  "dominant_signatures": ["...", "..."],
  "major_up_programs": ["...", "..."],
  "major_down_programs": ["...", "..."],
  "preliminary_mechanistic_interpretation": ["...", "..."],
  "unexpected_but_supported_findings": ["...", "..."],
  "evidence_notes": ["...", "..."],
  "confidence": "low|medium|high"
}
"""

REVIEWER_PROMPT = """
You are the Reviewer in a role-based multi-agent system for proteomics result interpretation.

Your role:
- critically evaluate the Analyzer output
- identify overstatements, unsupported claims, and missing caveats
- determine which findings are well-supported versus tentative
- help separate expected biology from potentially novel but plausible observations

Important interpretation rule:
Empty pathway sections can simply mean no significant enrichment was available for that category.
Do NOT criticize the analysis merely because Hallmark_up, Hallmark_down, GO_up, or GO_down is empty.

Strict rules:
1. Use the provided drug package as the primary source of truth.
2. Flag claims that are stronger than the evidence supports.
3. Prefer conservative phrasing when pathway evidence is sparse or absent.
4. Distinguish:
   - supported expected findings
   - plausible unexpected findings
   - weak / overinterpreted findings
5. Explicitly note missing evidence if a conclusion rests on too few proteins or only one data layer.

Return valid JSON only with this structure:
{
  "supported_expected_findings": ["...", "..."],
  "supported_unexpected_findings": ["...", "..."],
  "weak_or_overinterpreted_points": ["...", "..."],
  "recommended_caveats": ["...", "..."],
  "overall_assessment": "..."
}
"""

SUPERVISOR_PROMPT = """
You are the Supervisor in a role-based multi-agent system for proteomics result interpretation.

Your role:
- integrate the Analyzer and Reviewer outputs into a final concise scientific summary
- prioritize the strongest and most biologically meaningful conclusions
- preserve potentially interesting novel observations while clearly marking uncertainty
- produce a final summary suitable for a scientific manuscript supplement, drug ID card, or portal

Important interpretation rule:
Some pathway sections may be empty because no significant pathways were detected.
Do not penalize the final summary for that alone. Instead, integrate whatever evidence is available and state limitations clearly.

Strict rules:
1. Be concise, precise, and conservative.
2. Favor findings supported by coherent proteomic and/or pathway evidence.
3. Do not invent literature support.
4. Separate expected findings from unexpected but plausible findings.
5. Include follow-up hypotheses only when there is a clear biological rationale.
6. Keep the tone scientific and practical.

Return valid JSON only with this structure:
{
  "executive_summary": "...",
  "expected_findings": ["...", "..."],
  "unexpected_findings": ["...", "..."],
  "followup_hypotheses": [
    {
      "hypothesis": "...",
      "why_interesting": "...",
      "suggested_validation": "...",
      "priority": "high|medium|low"
    }
  ],
  "confidence": "low|medium|high",
  "caution_notes": ["...", "..."]
}
"""


## 4. Output templates

In [4]:

FINAL_SUMMARY_TEMPLATE: Dict[str, Any] = {
    "drug_name": "",
    "executive_summary": "",
    "expected_findings": [],
    "unexpected_findings": [],
    "followup_hypotheses": [],
    "confidence": "medium",
    "caution_notes": [],
}


def make_empty_final_summary(drug_name: str = "") -> Dict[str, Any]:
    template = deepcopy(FINAL_SUMMARY_TEMPLATE)
    template["drug_name"] = drug_name
    return template


## 5. Markdown card parsing and general utilities

In [5]:

def load_json(path: str | Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Dict[str, Any], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def save_markdown(text: str, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def load_drug_package(path: str | Path) -> Dict[str, Any]:
    pkg = load_json(path)
    return prepare_drug_package(pkg)


def ensure_json_response(text: str) -> Dict[str, Any]:
    text = (text or "").strip()
    if not text:
        raise ValueError("Model output was empty.")

    # Try direct JSON first.
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strip ```json fences if present.
    fenced = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    fenced = re.sub(r"\s*```$", "", fenced)
    try:
        return json.loads(fenced)
    except json.JSONDecodeError:
        pass

    # Extract the first JSON object.
    match = re.search(r"\{.*\}", fenced, flags=re.S)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    raise ValueError("Model output was not valid JSON.")


def split_markdown_sections(md_text: str) -> Dict[str, str]:
    pattern = re.compile(r"^##\s+(.+?)\s*$", re.M)
    matches = list(pattern.finditer(md_text))
    sections: Dict[str, str] = {}
    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(md_text)
        sections[match.group(1).strip()] = md_text[start:end].strip()
    return sections


def parse_basic_info_section(section_text: str) -> Dict[str, Any]:
    info: Dict[str, Any] = {}
    for line in section_text.splitlines():
        line = line.strip()
        match = re.match(r"-\s+\*\*(.+?):\*\*\s*(.*)$", line)
        if match:
            key = match.group(1).strip().lower().replace(" ", "_")
            info[key] = match.group(2).strip()
    return info


def parse_markdown_table(section_text: str) -> List[Dict[str, Any]]:
    text = section_text.strip()
    if not text or "_No data available_" in text or "No data available" in text:
        return []

    lines = [line.rstrip() for line in text.splitlines() if line.strip()]
    table_lines = [line for line in lines if line.lstrip().startswith("|")]
    if len(table_lines) < 2:
        return []

    header = [cell.strip() for cell in table_lines[0].strip().strip("|").split("|")]
    data_rows: List[Dict[str, Any]] = []

    for line in table_lines[2:]:
        values = [cell.strip() for cell in line.strip().strip("|").split("|")]
        if len(values) != len(header):
            continue
        row = dict(zip(header, values))
        data_rows.append(row)

    return data_rows


def split_pathways_by_nes(rows: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    up_rows: List[Dict[str, Any]] = []
    down_rows: List[Dict[str, Any]] = []

    for row in _clean_pathway_rows(rows):
        nes = row.get("NES")
        if isinstance(nes, (int, float)):
            if nes >= 0:
                up_rows.append(row)
            else:
                down_rows.append(row)

    return {"up": up_rows, "down": down_rows}


def build_drug_package_from_markdown_card(
    md_text: str,
    *,
    source_file: str = "",
    top_n_proteins: Optional[int] = None,
    top_n_pathways: Optional[int] = None,
    drop_crap: bool = True,
    drop_missing_gene: bool = True,
) -> Dict[str, Any]:
    sections = split_markdown_sections(md_text)
    pkg = make_empty_drug_package()

    basic = parse_basic_info_section(sections.get("1. Basic information", ""))
    pkg["basic_info"]["drug_name"] = basic.get("drug_name", "")
    pkg["basic_info"]["target"] = _split_semicolon_field(basic.get("target", ""))
    pkg["basic_info"]["pathway"] = _split_semicolon_field(basic.get("pathway", ""))
    pkg["basic_info"]["research_area"] = _split_semicolon_field(basic.get("research_area", ""))

    top_up = _clean_gene_rows(
        parse_markdown_table(sections.get("2. Top up-regulated genes", "")),
        drop_crap=drop_crap,
        drop_missing_gene=drop_missing_gene,
    )
    top_down = _clean_gene_rows(
        parse_markdown_table(sections.get("3. Top down-regulated genes", "")),
        drop_crap=drop_crap,
        drop_missing_gene=drop_missing_gene,
    )

    if top_n_proteins is not None:
        top_up = top_up[:top_n_proteins]
        top_down = top_down[:top_n_proteins]

    hallmark_rows = _clean_pathway_rows(
        parse_markdown_table(sections.get("4. Hallmark pathway profile", ""))
    )
    hallmark_split = split_pathways_by_nes(hallmark_rows)
    go_up_rows = _clean_pathway_rows(parse_markdown_table(sections.get("5. GO BP up-regulated pathways", "")))
    go_down_rows = _clean_pathway_rows(parse_markdown_table(sections.get("6. GO BP down-regulated pathways", "")))

    if top_n_pathways is not None:
        hallmark_split["up"] = hallmark_split["up"][:top_n_pathways]
        hallmark_split["down"] = hallmark_split["down"][:top_n_pathways]
        go_up_rows = go_up_rows[:top_n_pathways]
        go_down_rows = go_down_rows[:top_n_pathways]

    pkg["protein_summary"]["top_up"] = top_up
    pkg["protein_summary"]["top_down"] = top_down
    pkg["protein_summary"]["significant_counts"] = {
        "up": len(top_up),
        "down": len(top_down),
        "total": len(top_up) + len(top_down),
    }

    pkg["pathway_summary"]["hallmark_up"] = hallmark_split["up"]
    pkg["pathway_summary"]["hallmark_down"] = hallmark_split["down"]
    pkg["pathway_summary"]["go_bp_up"] = go_up_rows
    pkg["pathway_summary"]["go_bp_down"] = go_down_rows

    empty_sections: List[str] = []
    for section_name in ["hallmark_up", "hallmark_down", "go_bp_up", "go_bp_down"]:
        if not pkg["pathway_summary"][section_name]:
            empty_sections.append(section_name)
    pkg["pathway_summary"]["empty_sections"] = empty_sections

    pkg["metadata"]["source_file"] = source_file
    pkg["metadata"]["input_format"] = "drug_card_markdown"

    return prepare_drug_package(pkg)


def load_drug_package_from_markdown_card(
    path: str | Path,
    *,
    top_n_proteins: Optional[int] = None,
    top_n_pathways: Optional[int] = None,
    drop_crap: bool = True,
    drop_missing_gene: bool = True,
) -> Dict[str, Any]:
    path = Path(path)
    md_text = path.read_text(encoding="utf-8")
    return build_drug_package_from_markdown_card(
        md_text,
        source_file=path.name,
        top_n_proteins=top_n_proteins,
        top_n_pathways=top_n_pathways,
        drop_crap=drop_crap,
        drop_missing_gene=drop_missing_gene,
    )


## 6. LLM configuration and orchestrator

In [6]:

@dataclass
class LLMConfig:
    model: str = "gpt-5"
    temperature: float = 0.2
    max_retries: int = 2


class ProteomicsMASOrchestrator:
    """
    Runnable orchestration framework for a 3-agent proteomics interpretation pipeline.

    Pipeline:
        Analyzer -> Reviewer -> Supervisor
    """

    def __init__(
        self,
        api_key: Optional[str] = None,
        model: str = "gpt-5",
        temperature: float = 0.2,
        max_retries: int = 2,
        client: Optional[OpenAI] = None,
    ) -> None:
        self.config = LLMConfig(
            model=model,
            temperature=temperature,
            max_retries=max_retries,
        )

        if client is not None:
            self.client = client
        else:
            final_api_key = api_key or os.getenv("OPENAI_API_KEY")
            if OpenAI is None:
                raise ImportError("The 'openai' package is required to run the MAS agents. Install it with: pip install openai")
            if not final_api_key:
                raise ValueError("No API key provided. Pass api_key=... or set OPENAI_API_KEY.")
            self.client = OpenAI(api_key=final_api_key)

    def _call_model(self, system_prompt: str, payload: Dict[str, Any]) -> Dict[str, Any]:
        last_error: Optional[Exception] = None

        for attempt in range(self.config.max_retries + 1):
            try:
                response = self.client.chat.completions.create(
                    model=self.config.model,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": json.dumps(payload, ensure_ascii=False, indent=2)},
                    ],
                    temperature=self.config.temperature,
                    response_format={"type": "json_object"},
                )

                text = response.choices[0].message.content
                return ensure_json_response(text)

            except Exception as e:
                last_error = e
                if attempt >= self.config.max_retries:
                    break

        raise RuntimeError(f"Model call failed after retries: {last_error}")

    def run_analyzer(self, drug_package: Dict[str, Any]) -> Dict[str, Any]:
        drug_package = prepare_drug_package(drug_package)
        return self._call_model(ANALYZER_PROMPT, drug_package)

    def run_reviewer(self, drug_package: Dict[str, Any], analyzer_output: Dict[str, Any]) -> Dict[str, Any]:
        drug_package = prepare_drug_package(drug_package)
        payload = {"drug_package": drug_package, "analyzer_output": analyzer_output}
        return self._call_model(REVIEWER_PROMPT, payload)

    def run_supervisor(
        self,
        drug_package: Dict[str, Any],
        analyzer_output: Dict[str, Any],
        reviewer_output: Dict[str, Any],
    ) -> Dict[str, Any]:
        drug_package = prepare_drug_package(drug_package)
        payload = {
            "drug_package": drug_package,
            "analyzer_output": analyzer_output,
            "reviewer_output": reviewer_output,
        }
        return self._call_model(SUPERVISOR_PROMPT, payload)

    def run_for_one_drug(self, drug_package: Dict[str, Any]) -> Dict[str, Any]:
        drug_package = prepare_drug_package(drug_package)

        analyzer_output = self.run_analyzer(drug_package)
        reviewer_output = self.run_reviewer(drug_package, analyzer_output)
        supervisor_output = self.run_supervisor(drug_package, analyzer_output, reviewer_output)

        drug_name = drug_package.get("basic_info", {}).get("drug_name", "")
        final_summary = make_empty_final_summary(drug_name)
        final_summary.update(
            {
                "executive_summary": supervisor_output.get("executive_summary", ""),
                "expected_findings": supervisor_output.get("expected_findings", []),
                "unexpected_findings": supervisor_output.get("unexpected_findings", []),
                "followup_hypotheses": supervisor_output.get("followup_hypotheses", []),
                "confidence": supervisor_output.get("confidence", "medium"),
                "caution_notes": supervisor_output.get("caution_notes", []),
            }
        )

        return {
            "drug_name": drug_name,
            "input_package": drug_package,
            "analyzer_output": analyzer_output,
            "reviewer_output": reviewer_output,
            "supervisor_output": supervisor_output,
            "final_summary": final_summary,
        }

    def run_batch(self, drug_packages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        results: List[Dict[str, Any]] = []
        for pkg in drug_packages:
            drug_name = str(pkg.get("basic_info", {}).get("drug_name", "") or "")
            try:
                results.append(self.run_for_one_drug(pkg))
            except Exception as e:
                results.append({"drug_name": drug_name, "error": str(e)})
        return results

    def run_batch_from_json_files(self, json_paths: Iterable[str | Path]) -> List[Dict[str, Any]]:
        packages = [load_drug_package(path) for path in json_paths]
        return self.run_batch(packages)

    def run_batch_from_markdown_files(
        self,
        md_paths: Iterable[str | Path],
        *,
        top_n_proteins: Optional[int] = None,
        top_n_pathways: Optional[int] = None,
        drop_crap: bool = True,
        drop_missing_gene: bool = True,
    ) -> List[Dict[str, Any]]:
        packages = [
            load_drug_package_from_markdown_card(
                path,
                top_n_proteins=top_n_proteins,
                top_n_pathways=top_n_pathways,
                drop_crap=drop_crap,
                drop_missing_gene=drop_missing_gene,
            )
            for path in md_paths
        ]
        return self.run_batch(packages)

    def run_and_save_from_markdown_files(
        self,
        md_paths: Iterable[str | Path],
        out_dir: str | Path,
        *,
        top_n_proteins: Optional[int] = None,
        top_n_pathways: Optional[int] = None,
        drop_crap: bool = True,
        drop_missing_gene: bool = True,
        skip_existing: bool = False,
    ) -> List[Dict[str, Any]]:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        all_results: List[Dict[str, Any]] = []

        for path in md_paths:
            path = Path(path)
            pkg = load_drug_package_from_markdown_card(
                path,
                top_n_proteins=top_n_proteins,
                top_n_pathways=top_n_pathways,
                drop_crap=drop_crap,
                drop_missing_gene=drop_missing_gene,
            )
            drug_name = pkg["basic_info"]["drug_name"]
            safe_name = re.sub(r'[\\/:*?"<>|]+', "_", drug_name or path.stem)

            json_out = out_dir / f"{safe_name}.result.json"
            md_out = out_dir / f"{safe_name}.summary.md"

            if skip_existing and json_out.exists() and md_out.exists():
                all_results.append(load_json(json_out))
                continue

            try:
                result = self.run_for_one_drug(pkg)
                save_json(result, json_out)
                save_markdown(render_final_summary_markdown(result), md_out)
                all_results.append(result)
            except Exception as e:
                error_result = {"drug_name": drug_name, "source_file": path.name, "error": str(e)}
                save_json(error_result, out_dir / f"{safe_name}.error.json")
                all_results.append(error_result)

        return all_results


## 7. Markdown rendering

In [7]:

def render_final_summary_markdown(result: Dict[str, Any]) -> str:
    final_summary = result.get("final_summary", {})
    drug_name = final_summary.get("drug_name", result.get("drug_name", ""))
    executive_summary = final_summary.get("executive_summary", "")
    expected_findings = final_summary.get("expected_findings", [])
    unexpected_findings = final_summary.get("unexpected_findings", [])
    followup_hypotheses = final_summary.get("followup_hypotheses", [])
    confidence = final_summary.get("confidence", "")
    caution_notes = final_summary.get("caution_notes", [])

    lines: List[str] = []
    lines.append(f"# Drug summary: {drug_name}")
    lines.append("")
    lines.append("## Executive summary")
    lines.append(executive_summary if executive_summary else "Not available.")
    lines.append("")

    lines.append("## Findings consistent with expected biology")
    if expected_findings:
        for item in expected_findings:
            lines.append(f"- {item}")
    else:
        lines.append("- None listed.")
    lines.append("")

    lines.append("## Unexpected but plausible findings")
    if unexpected_findings:
        for item in unexpected_findings:
            lines.append(f"- {item}")
    else:
        lines.append("- None listed.")
    lines.append("")

    lines.append("## Follow-up hypotheses")
    if followup_hypotheses:
        for i, item in enumerate(followup_hypotheses, start=1):
            lines.append(f"### Hypothesis {i}")
            lines.append(f"- **Hypothesis:** {item.get('hypothesis', '')}")
            lines.append(f"- **Why interesting:** {item.get('why_interesting', '')}")
            lines.append(f"- **Suggested validation:** {item.get('suggested_validation', '')}")
            lines.append(f"- **Priority:** {item.get('priority', '')}")
            lines.append("")
    else:
        lines.append("- None listed.")
        lines.append("")

    lines.append("## Confidence")
    lines.append(confidence if confidence else "Not available.")
    lines.append("")

    lines.append("## Caution notes")
    if caution_notes:
        for note in caution_notes:
            lines.append(f"- {note}")
    else:
        lines.append("- None listed.")
    lines.append("")

    return "\n".join(lines)


## 8. Example: parse one markdown drug card

In [8]:
from pathlib import Path

input_dir = Path(r"E:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards")
example_md_path = input_dir / "Alcaftadine.md"

if example_md_path.exists():
    example_pkg = load_drug_package_from_markdown_card(
        example_md_path,
        top_n_proteins=25,
        top_n_pathways=15,
        drop_crap=True,
        drop_missing_gene=True,
    )

    print("Drug:", example_pkg["basic_info"]["drug_name"])
    print("Targets:", example_pkg["basic_info"]["target"])
    print("Empty pathway sections:", example_pkg["pathway_summary"]["empty_sections"])
    print("Top up proteins:", len(example_pkg["protein_summary"]["top_up"]))
    print("Top down proteins:", len(example_pkg["protein_summary"]["top_down"]))
    print("Hallmark up:", len(example_pkg["pathway_summary"]["hallmark_up"]))
    print("Hallmark down:", len(example_pkg["pathway_summary"]["hallmark_down"]))
    print("GO BP up:", len(example_pkg["pathway_summary"]["go_bp_up"]))
    print("GO BP down:", len(example_pkg["pathway_summary"]["go_bp_down"]))
else:
    print(f"File not found: {example_md_path}")

Drug: Alcaftadine
Targets: ['Histamine Receptor']
Empty pathway sections: ['go_bp_up', 'hallmark_down']
Top up proteins: 25
Top down proteins: 25
Hallmark up: 2
Hallmark down: 0
GO BP up: 0
GO BP down: 13


## 9. Example: save the standardized JSON package

In [9]:


outdir = Path(r"E:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards\normalized_examples")
outdir.mkdir(parents=True, exist_ok=True)

if "example_pkg" in globals():
    save_json(
        example_pkg,
        outdir / "Alcaftadine.package.json"
    )
    print("Saved to:", outdir / "Alcaftadine.package.json")


Saved to: E:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards\normalized_examples\Alcaftadine.package.json


## 10. Example: run one drug through the MAS

In [10]:

# # Requires OPENAI_API_KEY in your environment.

# # ===== API =====
# api_key = os.getenv("OPENAI_API_KEY")
# orchestrator = ProteomicsMASOrchestrator(
#     api_key=api_key,
#     model="gpt-5",
#     temperature=0.2
# )

# # ===== 运行 MAS =====
# result = orchestrator.run_for_one_drug(example_pkg)

# # ===== 输出路径（你指定的路径）=====
# outdir = Path(r"C:\Users\jymbc\Desktop\drug portal\AIresults")
# outdir.mkdir(parents=True, exist_ok=True)

# # ===== 文件名 =====
# drug_name = "Alcaftadine"

# # ===== 保存 JSON =====
# save_json(
#     result,
#     outdir / f"{drug_name}.result.json"
# )

# # ===== 保存 Markdown =====
# save_markdown(
#     render_final_summary_markdown(result),
#     outdir / f"{drug_name}.summary.md"
# )

# print("Saved one-drug MAS outputs to:", outdir)


In [12]:
from openai import AzureOpenAI
import os
import getpass
from pathlib import Path

key = getpass.getpass(prompt="Enter your API key: ")

endpoint = "https://meyer-east-2.openai.azure.com/"

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=key,
    api_version="2024-12-01-preview"
)

orchestrator = ProteomicsMASOrchestrator(
    client=client,          # 关键：直接传入 Azure client
    model="gpt-5",
    temperature=1
)

result = orchestrator.run_for_one_drug(example_pkg)

outdir = Path(r"C:\Users\jymbc\Desktop\drug portal\AIresults")
outdir.mkdir(parents=True, exist_ok=True)

drug_name = example_pkg["basic_info"]["drug_name"]

save_json(result, outdir / f"{drug_name}.result.json")
save_markdown(
    render_final_summary_markdown(result),
    outdir / f"{drug_name}.summary.md"
)

print("Saved one-drug MAS outputs.")

Enter your API key:  ········


Saved one-drug MAS outputs.


## 11. Example: batch-run a whole folder of markdown drug cards

In [11]:
# from openai import AzureOpenAI
# from pathlib import Path
# import getpass
# import os

# key = getpass.getpass(prompt="Enter your API key: ")

# endpoint = "https://meyer-east-2.openai.azure.com/"
# client = AzureOpenAI(
#     azure_endpoint=endpoint,
#     api_key=key,
#     api_version="2024-12-01-preview"
# )

# orchestrator = ProteomicsMASOrchestrator(
#     client=client,
#     model="gpt-5",
#     temperature=1
# )

# md_dir = Path(r"E:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards")
# out_dir = Path(r"E:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards_MAS_results")
# out_dir.mkdir(parents=True, exist_ok=True)

# md_paths = sorted(md_dir.glob("*.md"))

# all_results = []

# for i, md_path in enumerate(md_paths, start=1):
#     print(f"[{i}/{len(md_paths)}] Processing: {md_path.name}")
    
#     try:
#         results = orchestrator.run_and_save_from_markdown_files(
#             [md_path],   # 一次只跑一个
#             out_dir=out_dir,
#             top_n_proteins=30,
#             top_n_pathways=15,
#             drop_crap=True,
#             drop_missing_gene=True,
#             skip_existing=True
#         )
#         all_results.extend(results)
        
#         if results and "error" in results[0]:
#             print(f"  -> Failed: {results[0]['error']}")
#         else:
#             print("  -> Success")
    
#     except Exception as e:
#         print(f"  -> Unexpected error: {e}")
#         all_results.append({
#             "drug_name": md_path.stem,
#             "error": str(e)
#         })

# print("\nBatch run completed.")
# print(f"Total files processed: {len(md_paths)}")
# print(f"Output folder: {out_dir}")

# 12. Example: batch-run per drug of markdown drug cards

In [13]:

# =========================
# 1. Azure client
# =========================
key = getpass.getpass(prompt="Enter your API key: ")
endpoint = "https://meyer-east-2.openai.azure.com/"

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=key,
    api_version="2024-12-01-preview"
)

# =========================
# 2. Orchestrator
# =========================
orchestrator = ProteomicsMASOrchestrator(
    client=client,
    model="gpt-5",
    temperature=1
)

# =========================
# 3. Input / Output paths
# =========================
input_dir = Path(r"E:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards")
output_dir = Path(r"C:\Users\jymbc\Desktop\drug portal\AIresults")
output_dir.mkdir(parents=True, exist_ok=True)

# =========================
# 4. Safe filename helper
# =========================
def safe_filename(name: str) -> str:
    return re.sub(r'[\\/:*?"<>|]+', "_", str(name)).strip()

# =========================
# 5. Batch run
# =========================
md_files = sorted(input_dir.glob("*.md"))

print(f"Found {len(md_files)} markdown files.")

for md_file in md_files:
    try:
        example_pkg = load_drug_package_from_markdown_card(
            md_file,
            top_n_proteins=25,
            top_n_pathways=15,
            drop_crap=True,
            drop_missing_gene=True,
        )

        drug_name = example_pkg["basic_info"].get("drug_name", md_file.stem)
        safe_name = safe_filename(drug_name)

        result = orchestrator.run_for_one_drug(example_pkg)

        save_json(result, output_dir / f"{safe_name}.result.json")
        save_markdown(
            render_final_summary_markdown(result),
            output_dir / f"{safe_name}.summary.md"
        )

        print(f"Saved: {safe_name}")

    except Exception as e:
        print(f"Failed: {md_file.name} -> {e}")

Enter your API key:  ········


Found 170 markdown files.
Saved: 6-Mercaptopurine
Saved: Abemaciclib (methanesulfonate)
Saved: Acetophenazine (dimaleate)
Saved: Adiphenine (hydrochloride)
Saved: Alcaftadine
Saved: Alizapride (hydrochloride)
Saved: Almotriptan (malate)
Saved: Ambroxol
Saved: Amcinonide
Saved: Amiodarone (hydrochloride)
Saved: Amisulpride
Saved: Amodiaquine (dihydrochloride)
Saved: Anamorelin
Saved: Andrographolide
Saved: Aniracetam
Saved: Antazoline (hydrochloride)
Saved: Antipyrine
Saved: Asciminib
Saved: Atazanavir (sulfate)
Saved: Atorvastatin (hemicalcium salt)
Saved: ATP (dimagnesium)
Saved: Azatadine (dimaleate)
Saved: Azelastine (hydrochloride)
Saved: Baicalein
Saved: Balsalazide
Saved: Belzutifan
Saved: Benzamil (hydrochloride)
Saved: Benztropine (mesylate)
Saved: Bepotastine
Saved: Betahistine
Saved: Betamethasone dipropionate
Saved: Betamethasone disodium phosphate
Saved: Betamethasone valerate
Saved: Betazole
Saved: Bicyclol
Saved: Binimetinib
Saved: Bromhexine (hydrochloride)
Saved: Budeso